# 手搓 Transformer: 注入 Hugging Face MarianMT 工业级预训练权重

我们将把 `Helsinki-NLP/opus-mt-en-de` 的官方权重注入到我们自己手写的 `Transformer` 实例中，实现一个英译德的翻译系统。

In [1]:
import torch

from test_translation import (
    MODEL_NAME,
    build_models,
    load_weights_from_marian,
    translate,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

d:\Python\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
d:\Python\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda


## 1. 下载模型和分词器

In [2]:
print(f"正在加载 {MODEL_NAME}...")
tokenizer, marian_model, my_transformer = build_models()

vocab_size = marian_model.model.shared.weight.shape[0]
print("分词器词表大小:", vocab_size)
print("官方模型架构特征:")
print(f"d_model: {marian_model.config.d_model}")
print(f"Encoder Layers: {marian_model.config.encoder_layers}")
print(f"Decoder Layers: {marian_model.config.decoder_layers}")
print(f"Attention Heads: {marian_model.config.encoder_attention_heads}")

正在加载 Helsinki-NLP/opus-mt-en-de...


d:\Python\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


分词器词表大小: 58101
官方模型架构特征:
d_model: 512
Encoder Layers: 6
Decoder Layers: 6
Attention Heads: 8


## 2. 实例化手写的 Transformer 并进行架构微调

根据 MarianMT 的配置创建我们的模型。`transformer.py` 已在 `MutiHeadAttention` 内部集成 `out_proj`，这里不再需要外部包装补丁。

In [3]:
print("手写模型实例化完成（使用 transformer.py 内置 out_proj）")

手写模型实例化完成（使用 transformer.py 内置 out_proj）


## 3. 权重手术 (Weight Surgery)

这一步非常硬核，我们将把官方复杂的 state_dict 逐层“粘贴”到我们的 `my_transformer` 实例中。

In [4]:
load_weights_from_marian(marian_model, my_transformer)
print("[OK] 权重注入成功！")

[OK] 权重注入成功！


## 4. 自回归解码推理函数

编写推理循环：每次生成一个词，直到输出 `<eos>` (end of sentence) 为止。

In [5]:
# 推理函数已经在 test_translation.py 里实现并导入，这里直接复用。

## 5. 进行端到端测试

In [8]:
for text in [
    "This is a great movie! I really enjoyed watching it.",
    "The acting was terrible, and the plot made no sense.",
    "Deep learning has completely revolutionized natural language processing.",
    "Funny this is my test sentence"
]:
    translated = translate(text, tokenizer, my_transformer)
    print(f"EN: {text}")
    print(f"DE: {translated}\n")

EN: This is a great movie! I really enjoyed watching it.
DE: Das ist ein toller Film! Ich habe ihn wirklich genossen.

EN: The acting was terrible, and the plot made no sense.
DE: Das Schauspiel war schrecklich, und die Handlung ergab keinen Sinn.

EN: Deep learning has completely revolutionized natural language processing.
DE: Deep Learning hat die natürliche Sprachverarbeitung völlig revolutioniert.

EN: Funny this is my test sentence
DE: Komisch, das ist mein Testsatz

